In [0]:
import json
import os
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [0]:
bronze_df = spark.table('workspace.default.cricket_api_project')
raw_json = bronze_df.select('raw_json').collect()[0]['raw_json']
api_data = json.loads(raw_json)

print(type(api_data))
print(api_data)
print(len(api_data))
matches = api_data[0]
print(type(matches))
print(matches)


#display(bronze_df)
#display(bronze_df.select('raw_json').collect()[0]['raw_json'])
#display(bronze_df.select())



In [0]:

silver_rows = []

for match in api_data:
    silver_rows.append({
        "userId": match.get("userId"),
        "id": match.get("id"),
        "title": match.get("title"),
        "body": match.get("body")
    })

print("silver rows appended:", len(silver_rows))


In [0]:
silver_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("body", StringType(), True)
])


silver_df = spark.createDataFrame(silver_rows,silver_schema)\
    .withColumn("ingestion_time",current_timestamp())

display(silver_df)
#silver_df.write.mode("overwrite").saveAsTable("")

In [0]:
silver_df.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema","true")\
    .saveAsTable("workspace.default.cricket_silver_api")

print("silver tabel created successfully")




In [0]:
%sql
select * from workspace.default.cricket_silver_api

In [0]:
# Save the notebook as a .dbc or .ipynb file and push to git
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_name = notebook_path.split('/')[-1] + ".ipynb"
local_path = f"/tmp/{notebook_name}"

# Export notebook to local file
dbutils.notebook.export(notebook_path, local_path, format="SOURCE")

# Git commands to push notebook (requires git setup in cluster)
os.system(f"git add {local_path}")
os.system(f"git commit -m 'Add notebook {notebook_name}'")
os.system("git push")